# CredAIR — Análise de Risco de Crédito

**Objetivo:** usar o histórico de propostas de crédito da CredAIR para entender os fatores ligados à inadimplência e construir um modelo de **Regressão Logística** que ajude a identificar propostas de maior risco.

> Os dados são **sintéticos** e foram criados para fins educacionais.

### Roteiro do notebook

| Seção | Conteúdo | Camada de dados |
|---|---|---|
| 1 | Entendimento do problema | 🥉 bronze |
| 2 | Diagnóstico de qualidade dos dados | 🥉 bronze |
| 3 | Limpeza e preparação | 🥉 → 🥈 prata |
| 4 | Análise exploratória | 🥈 prata |
| 5 | Engenharia de atributos | 🥈 → 🥇 ouro |
| 6 | Preparação para modelagem | 🥇 ouro |
| 7 | Modelagem | 🥇 ouro |
| 8 | Avaliação e interpretação | — |
| 9 | Conclusão | — |

Ao longo do notebook, cada decisão importante aparece num bloco **🧭 Decisão**, com o que foi feito, por quê e qual alternativa foi descartada.

## 0. Configuração do ambiente

Importamos as bibliotecas e fixamos uma **semente aleatória** (`RANDOM_STATE`). Com ela, a divisão treino/teste e o treinamento produzem sempre os mesmos resultados, o que torna a análise **reprodutível**.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

RANDOM_STATE = 42

# Raiz do projeto: funciona tanto no VS Code (raiz = pasta do projeto)
# quanto no Jupyter aberto dentro de notebooks/
RAIZ = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
BRONZE = RAIZ / "data" / "bronze" / "dataset.csv"
PRATA = RAIZ / "data" / "silver" / "credito_limpo.parquet"
OURO = RAIZ / "data" / "gold" / "credito_modelagem.parquet"
FIGURAS = RAIZ / "reports" / "figures"
FIGURAS.mkdir(parents=True, exist_ok=True)

# Estilo dos gráficos: fundo limpo, grade discreta e duas cores fixas para as classes
AZUL, LARANJA, CINZA = "#2a78d6", "#eb6834", "#898781"
CORES_CLASSE = {0: AZUL, 1: LARANJA}
sns.set_theme(style="whitegrid", rc={"grid.color": "#e1e0d9", "axes.edgecolor": "#c3c2b7"})
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.3f}".format)


def salvar(fig, nome):
    # Salva a figura em reports/figures para usar na apresentação
    fig.savefig(FIGURAS / f"{nome}.png", dpi=120, bbox_inches="tight")

## 1. Entendimento do problema

### 1.1 Carregando os dados brutos (camada bronze)

Lemos o arquivo **exatamente como foi recebido**. Nenhuma alteração é feita nesta etapa.

In [2]:
bronze = pd.read_csv(BRONZE)
print(f"Linhas: {bronze.shape[0]:,} | Colunas: {bronze.shape[1]}")
bronze.head()

Linhas: 5,000 | Colunas: 11


,idade,estado,escolaridade,renda_mensal,tempo_emprego_anos,score_credito,valor_solicitado,prazo_meses,finalidade,dividas_ativas,inadimplente
0,52,MS,Ensino Superior,"2,911.820",14,690,"5,934.190",60,Viagem,1,0
1,49,RR,Ensino Médio,"3,176.430",5,676,"6,791.380",24,Outros,2,0
2,48,AL,Ensino Superior,"3,429.400",23,558,"14,666.780",24,Outros,5,1
3,59,GO,Ensino Fundamental,"10,096.700",2,739,"19,053.240",24,Reforma,0,0
4,37,RJ,Ensino Fundamental,"3,033.250",24,797,"6,171.310",6,Educação,0,1


### 1.2 Dicionário de dados

| Coluna | Tipo | Papel | Descrição |
|---|---|---|---|
| `idade` | numérica | explicativa | Idade do solicitante na proposta |
| `estado` | categórica | explicativa | UF de residência |
| `escolaridade` | categórica **ordinal** | explicativa | Nível de escolaridade declarado |
| `renda_mensal` | numérica | explicativa | Renda mensal declarada (R$) |
| `tempo_emprego_anos` | numérica | explicativa | Anos no emprego atual |
| `score_credito` | numérica | explicativa | Pontuação de crédito do solicitante |
| `valor_solicitado` | numérica | explicativa | Valor do empréstimo (R$) |
| `prazo_meses` | numérica discreta | explicativa | Prazo de pagamento (meses) |
| `finalidade` | categórica | explicativa | Finalidade do empréstimo |
| `dividas_ativas` | numérica discreta | explicativa | Quantidade de dívidas em aberto |
| **`inadimplente`** | **binária (0/1)** | **resposta** | **1 = a proposta resultou em inadimplência** |

> **Divergência no enunciado:** o texto cita o arquivo `credito.csv`, mas o arquivo fornecido se chama `dataset.csv`. Mantivemos o nome original para preservar a rastreabilidade do dado bruto.

### 1.3 Variável resposta e tipo de problema

- **Variável resposta:** `inadimplente`. Indica se a proposta, **depois de acompanhada ao longo do tempo**, resultou em falta de pagamento (`1`) ou não (`0`).
- **Por que é Machine Learning supervisionado:** temos exemplos históricos **com a resposta conhecida**. O modelo aprende a relação entre as características da proposta e o resultado observado.
- **Por que é classificação, e não regressão:** a resposta é uma **categoria** (inadimplente ou não), e não uma quantidade contínua. Mais precisamente, é uma **classificação binária**.
- **O que o modelo vai entregar:** a Regressão Logística estima a **probabilidade** de inadimplência, um número entre 0 e 1. Para virar uma decisão (aprovar, recusar ou analisar), essa probabilidade é comparada com um **limiar**, que vamos discutir na seção 8.

### 1.4 O custo dos erros

No crédito, os dois tipos de erro **não custam o mesmo**:

| Erro | O que acontece | Custo para a CredAIR |
|---|---|---|
| **Falso negativo:** o modelo diz "bom pagador", mas a pessoa fica inadimplente | Empréstimo aprovado e não pago | **Perda do capital emprestado** (alto) |
| **Falso positivo:** o modelo diz "risco", mas a pessoa pagaria | Proposta recusada sem necessidade | Perda dos juros que seriam recebidos e do cliente (moderado) |

Por isso, na avaliação, **não basta olhar a acurácia**. Vamos dar atenção especial ao **recall da classe inadimplente**, ou seja, a fração dos maus pagadores que o modelo consegue identificar.

In [3]:
contagem = bronze["inadimplente"].value_counts().sort_index()
pd.DataFrame({
    "quantidade": contagem,
    "proporção": (contagem / contagem.sum()).map("{:.1%}".format),
}).rename(index={0: "0 - adimplente", 1: "1 - inadimplente"})

,quantidade,proporção
inadimplente,,
0 - adimplente,3280,65.6%
1 - inadimplente,1720,34.4%


**Primeira leitura:** cerca de **34%** das propostas resultaram em inadimplência. As classes são **moderadamente desbalanceadas**. Um "modelo" que dissesse sempre "adimplente" acertaria cerca de 66% das vezes sem aprender nada. Por isso a acurácia sozinha engana, e vamos comparar o modelo com essa referência na seção 7.

## 2. Diagnóstico de qualidade dos dados

Antes de corrigir qualquer coisa, fazemos um **inventário dos problemas**.

> 🧭 **Decisão: diagnosticar antes de tratar.** Separar as duas etapas evita "consertar" algo sem entender a causa e permite justificar cada tratamento com evidências. A alternativa descartada foi limpar enquanto explorávamos, o que mistura descobertas com correções e dificulta auditar.

### 2.1 Tipos e valores ausentes

In [4]:
pd.DataFrame({
    "tipo": bronze.dtypes.astype(str),
    "nulos": bronze.isna().sum(),
    "valores_distintos": bronze.nunique(),
})

,tipo,nulos,valores_distintos
idade,int64,0,73
estado,str,0,85
escolaridade,str,0,4
renda_mensal,float64,0,4921
tempo_emprego_anos,int64,0,32
score_credito,int64,0,428
valor_solicitado,float64,0,4937
prazo_meses,int64,0,7
finalidade,str,0,6
dividas_ativas,int64,0,8


Não há valores nulos **explícitos**. Mas isso não garante que os dados estejam corretos: um problema pode aparecer como valor inválido em vez de vazio. Vamos verificar.

### 2.2 Estatísticas descritivas

In [5]:
bronze.describe().T

,count,mean,std,min,25%,50%,75%,max
idade,"5,000.000",47.767,31.646,18.000,32.000,46.000,61.000,750.000
renda_mensal,"5,000.000","5,228.634","3,968.478",100.000,"2,697.320","4,146.300","6,458.877","44,814.190"
tempo_emprego_anos,"5,000.000",12.662,7.518,1.000,6.000,13.000,19.000,74.000
score_credito,"5,000.000",652.481,84.817,335.000,594.000,653.000,710.000,850.000
valor_solicitado,"5,000.000","14,264.657","8,268.261","1,721.480","8,467.170","12,365.630","17,764.885","50,000.000"
prazo_meses,"5,000.000",29.922,15.483,6.000,18.000,24.000,36.000,60.000
dividas_ativas,"5,000.000",1.900,1.865,0.000,0.000,1.000,3.000,7.000
inadimplente,"5,000.000",0.344,0.475,0.000,0.000,0.000,1.000,1.000


Alguns sinais aparecem já na tabela:
- `idade` chega a **750 anos**.
- `tempo_emprego_anos` chega a **74 anos**.
- `renda_mensal` tem mínimo de **R$ 100**.

### 2.3 Duplicidades

In [6]:
n_dup = bronze.duplicated().sum()
print(f"Linhas duplicadas (idênticas em todas as colunas): {n_dup}")
bronze[bronze.duplicated(keep=False)].sort_values(list(bronze.columns)).head(6)

Linhas duplicadas (idênticas em todas as colunas): 19


,idade,estado,escolaridade,renda_mensal,tempo_emprego_anos,score_credito,valor_solicitado,prazo_meses,finalidade,dividas_ativas,inadimplente
813,20,RN,Ensino Superior,"7,393.680",13,826,"20,006.350",36,Reforma,2,0
3723,20,RN,Ensino Superior,"7,393.680",13,826,"20,006.350",36,Reforma,2,0
2748,23,SC,Ensino Médio,"5,657.970",3,718,"6,468.510",36,Reforma,1,0
4726,23,SC,Ensino Médio,"5,657.970",3,718,"6,468.510",36,Reforma,1,0
2177,25,AC,Ensino Médio,"4,663.760",10,715,"12,020.380",36,Veículo,0,0
3089,25,AC,Ensino Médio,"4,663.760",10,715,"12,020.380",36,Veículo,0,0


Há linhas **idênticas em todas as 11 colunas**, inclusive renda e valor com centavos. A chance de duas propostas diferentes coincidirem em tudo isso é praticamente nula. Isso indica **registro duplicado**, e não dois clientes iguais.

### 2.4 Categorias: inconsistências de formatação

In [7]:
for col in ["estado", "escolaridade", "finalidade"]:
    valores = bronze[col]
    com_espaco = (valores != valores.str.strip()).sum()
    print(f"{col:13s} | categorias distintas: {valores.nunique():3d} | "
          f"após remover espaços: {valores.str.strip().nunique():3d} | registros com espaços: {com_espaco}")

print("\nExemplos:", bronze.loc[bronze["estado"] != bronze["estado"].str.strip(), "estado"].map(repr).unique()[:6])

estado        | categorias distintas:  85 | após remover espaços:  27 | registros com espaços: 101
escolaridade  | categorias distintas:   4 | após remover espaços:   4 | registros com espaços: 0
finalidade    | categorias distintas:   6 | após remover espaços:   6 | registros com espaços: 0

Exemplos: <ArrowStringArray>
['' DF '', '' PA '', '' BA '', '' PI '', ''  CE '', '' AM  '']
Length: 6, dtype: str


A coluna `estado` tem espaços extras (ex.: `' MG '`). Para o computador, `' MG '` e `'MG'` são categorias **diferentes**, o que criaria estados "fantasmas" na análise. As outras colunas categóricas estão corretas.

### 2.5 Regras de domínio: valores impossíveis ou suspeitos

Comparamos os dados com **regras do mundo real**.

In [8]:
idade_valida = bronze["idade"].between(18, 100)

regras = {
    "idade fora de 18–100 anos": ~idade_valida,
    "tempo de emprego ≥ idade": bronze["tempo_emprego_anos"] >= bronze["idade"],
    "começou a trabalhar antes dos 14 anos (idade válida)":
        idade_valida & (bronze["tempo_emprego_anos"] > bronze["idade"] - 14),
    "renda mensal < R$ 600": bronze["renda_mensal"] < 600,
    "score fora de 0–1000": ~bronze["score_credito"].between(0, 1000),
    "valor solicitado ≤ 0": bronze["valor_solicitado"] <= 0,
    "dívidas ativas < 0": bronze["dividas_ativas"] < 0,
}
pd.DataFrame({"registros": {k: int(v.sum()) for k, v in regras.items()}})

,registros
idade fora de 18–100 anos,15
tempo de emprego ≥ idade,8
começou a trabalhar antes dos 14 anos (idade válida),696
renda mensal < R$ 600,10
score fora de 0–1000,0
valor solicitado ≤ 0,0
dívidas ativas < 0,0


In [9]:
print("Idades fora do intervalo:", sorted(bronze.loc[~idade_valida, "idade"]))
print("Rendas abaixo de R$ 600:  ", sorted(bronze.loc[bronze["renda_mensal"] < 600, "renda_mensal"]))
print("\nValores que se repetem muito (possíveis valores 'padrão' ou tetos):")
print("  renda = 900:    ", (bronze["renda_mensal"] == 900).sum(), "registros")
print("  renda = 30.000: ", (bronze["renda_mensal"] == 30000).sum(), "registros")
print("  valor = 50.000: ", (bronze["valor_solicitado"] == 50000).sum(), "registros")

Idades fora do intervalo: [210, 280, 300, 340, 360, 430, 440, 500, 540, 660, 670, 680, 700, 710, 750]
Rendas abaixo de R$ 600:   [100.0, 150.0, 200.0, 250.0, 300.0, 350.0, 400.0, 450.0, 500.0, 550.0]

Valores que se repetem muito (possíveis valores 'padrão' ou tetos):
  renda = 900:     34 registros
  renda = 30.000:  8 registros
  valor = 50.000:  32 registros


In [10]:
# Onde estão os casos de "começou a trabalhar antes dos 14 anos"?
inconsistente = regras["começou a trabalhar antes dos 14 anos (idade válida)"]
faixa = pd.cut(bronze["idade"], [17, 25, 35, 45, 100], labels=["18–25", "26–35", "36–45", "46+"])
pd.DataFrame({
    "casos": inconsistente.groupby(faixa, observed=True).sum(),
    "% da faixa": inconsistente.groupby(faixa, observed=True).mean().map("{:.0%}".format),
})

,casos,% da faixa
idade,,
18–25,351,50%
26–35,314,36%
36–45,25,3%
46+,6,0%


### 2.6 Resumo do diagnóstico

| # | Problema | Registros | Evidência | Tratamento proposto (seção 3) |
|---|---|---|---|---|
| P1 | Espaços extras em `estado` | ~100 | `' MG '` ≠ `'MG'` | Remover espaços e padronizar maiúsculas |
| P2 | Linhas duplicadas | 19 | Idênticas em todas as colunas | Remover, mantendo a primeira ocorrência |
| P3 | Idade impossível (210 a 750) | 15 | Fora de 18–100 | Marcar como **ausente** (imputar no pipeline) |
| P4 | Tempo de emprego ≥ idade (30 a 74 anos) | 8 | Fisicamente impossível | Marcar como **ausente** |
| P5 | Renda de R$ 100 a R$ 550, de 50 em 50 | 10 | Sequência artificial, longe do restante | Marcar como **ausente** |
| P6 | Início de trabalho antes dos 14 anos | ~690 | Concentrado entre 18 e 35 anos | **Manter** e registrar como limitação |
| P7 | Valores repetidos: renda R$ 900 e R$ 30.000; valor R$ 50.000 | 34 / 8 / 32 | Parecem piso ou teto do formulário | **Manter**, porque são valores plausíveis |

As justificativas de cada tratamento estão na seção 3.